In [ ]:
%%capture
!pip install -U bitsandbytes
!pip install openai selenium python-docx google-colab-selenium[undetected] google-generativeai
!pip install python-docx langchain

In [ ]:
import openai
from openai import OpenAI
import os
from docx import Document
import base64
import requests
import io
from PIL import Image
import google_colab_selenium as gs
from selenium import webdriver
from selenium.webdriver.chrome.service import Service as ChromeService
from selenium.webdriver.chrome.options import Options
from google.colab import userdata
import re

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name_or_path = "Qwen/Qwen2.5-7B-Instruct"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)
model = AutoModelForCausalLM.from_pretrained(
    model_name_or_path,
    torch_dtype="auto",
    device_map="auto",
    load_in_4bit=True,
)

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

...

model-00002-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/3.56G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

In [ ]:
# @title Functions to Call
def generate_split_layout_slide1(
    left_bg_color="#F8F8F8",
    left_title="Báo cáo VNPT-AI về bài toán Function Calling",
    left_title_color="#2C4B7D",
    left_subtitle="Báo cáo về các bài báo đã tìm hiểu được",
    left_subtitle_color="#333",
    left_dots="•••••",
    left_dots_color="#2C4B7D",
    right_bg_color="#F8F8F8",
    right_image_src="path-to-your-image/LLM-image.png",
    right_image_alt="LLM Image",
    decor_bg_color="#2C4B7D",
    font_family="Arial, sans-serif"
):
    """
    Generate a split-layout HTML slide with customizable parameters.

    :param left_bg_color: Background color of the left section.
    :param left_title: Title text in the left section.
    :param left_title_color: Color of the title in the left section.
    :param left_subtitle: Subtitle text in the left section.
    :param left_subtitle_color: Color of the subtitle in the left section.
    :param left_dots: Text or symbols for the dots section in the left section.
    :param left_dots_color: Color of the dots in the left section.
    :param right_bg_color: Background color of the right section.
    :param right_image_src: Source path for the image in the right section.
    :param right_image_alt: Alt text for the image in the right section.
    :param decor_bg_color: Background color of the decorative container.
    :param font_family: Font family for the slide content.
    :return: A string containing the HTML code.
    """

    html_code = f"""<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <!-- Bootstrap CSS -->
    <link href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.0/dist/css/bootstrap.min.css" rel="stylesheet">
    <style>
        body, html {{
            margin: 0;
            padding: 0;
            height: 100%;
            font-family: {font_family};
            background-color: #E0E0E0;
        }}
        .left-section {{
            background-color: {left_bg_color};
            display: flex;
            flex-direction: column;
            justify-content: center;
            padding: 60px;
            box-sizing: border-box;
            z-index: 1;
        }}
        .left-section .title {{
            font-size: 48px;
            font-weight: bold;
            color: {left_title_color};
            margin-bottom: 20px;
        }}
        .left-section .subtitle {{
            font-size: 24px;
            color: {left_subtitle_color};
            margin-bottom: 40px;
        }}
        .left-section .dots {{
            font-size: 36px;
            color: {left_dots_color};
        }}
        .right-section {{
            display: flex;
            align-items: center;
            justify-content: center;
            padding: 28px;
            background-color: {right_bg_color};
            border-radius: 10px;
            z-index: 1;
        }}
        .right-section img {{
            max-width: 100%;
            height: auto;
            border-radius: 10px;
        }}
        .container::before {{
            content: '';
            position: absolute;
            width: 100%;
            height: 50%;
            background-color: {decor_bg_color};
            top: 50%;
            left: 0;
            z-index: -1;
        }}
        .decor-container {{
            position: absolute;
            width: 100%;
            height: 30%;
            background-color: {decor_bg_color};
            bottom: 0;
            left: 0;
            z-index: 0;
        }}
    </style>
</head>
<body>
    <div class='container d-flex flex-row justify-content-center align-items-center position-relative h-100'>
        <div class="row w-100">
            <div class="col-md-6 left-section">
                <div class="title">{left_title}</div>
                <div class="subtitle">{left_subtitle}</div>
                <div class="dots">{left_dots}</div>
            </div>
            <div class="col-md-4 offset-md-1 right-section">
                <img src="{right_image_src}" alt="{right_image_alt}">
            </div>
        </div>
    </div>
    <div class="decor-container"></div>

    <!-- Bootstrap JS -->
    <script src="https://cdn.jsdelivr.net/npm/bootstrap@5.3.0/dist/js/bootstrap.bundle.min.js"></script>
</body>
</html>"""

    return html_code

def generate_split_layout_slide2(
    title="Split Layout HTML Slide",
    left_bg_color="#2C4B7D",
    left_text_color="white",
    left_number="01",
    right_bg_color="#F0F0F0",
    image_placeholder_text_1="LLM (Large Language Model) logo Placeholder",
    image_placeholder_text_2="LLM Agent Pipeline Placeholder",
    bottom_title="UltraTool",
    bottom_title_color="#2C4B7D",
    bottom_description="UltraTool cải thiện gọi hàm nhờ lập kế hoạch, xây dựng tools theo yêu cầu phức tạp, và tạo Tool mới khi cần thiết nếu các tool hiện có không đáp ứng được.",
    bottom_description_color="#333",
    divider_color="#000",
    font_family="Arial, sans-serif"
):
    """
    Generate a split layout HTML slide with customizable parameters.

    :param title: The title of the HTML document.
    :param left_bg_color: Background color of the left section.
    :param left_text_color: Text color of the left section.
    :param left_number: The number displayed in the left section.
    :param right_bg_color: Background color of the right section.
    :param image_placeholder_text_1: Text displayed in the first image placeholder.
    :param image_placeholder_text_2: Text displayed in the second image placeholder.
    :param bottom_title: Title displayed in the bottom right section.
    :param bottom_title_color: Color of the bottom title.
    :param bottom_description: Description text in the bottom right section.
    :param bottom_description_color: Color of the bottom description text.
    :param divider_color: Color of the divider in the bottom right section.
    :param font_family: Font family for the slide content.
    :return: A string containing the HTML code.
    """

    html_code = f"""<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <!-- Bootstrap CSS -->
    <link href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.0/dist/css/bootstrap.min.css" rel="stylesheet">
    <style>
        body, html {{
            margin: 0;
            padding: 0;
            height: 100%;
            font-family: {font_family};
            background-color: {right_bg_color};
        }}
        .left-section {{
            background-color: {left_bg_color};
            color: {left_text_color};
            display: flex;
            justify-content: top;
            align-items: center;
            flex-direction: column;
            padding: 20px;
            z-index: 1;
        }}
        .left-section .number {{
            font-size: 120px;
            font-weight: bold;
            margin-top: 80px;
        }}
        .right-section {{
            background-color: {right_bg_color};
            display: flex;
            flex-direction: column;
            justify-content: space-evenly;
            padding: 20px;
            z-index: 2;
        }}
        .image-placeholder {{
            background-color: #ccc;
            display: flex;
            align-items: center;
            justify-content: center;
            border: 1px solid #000;
            height: 250px;
        }}
        .bottom-right {{
            background-color: white;
            padding: 50px;
            display: flex;
            justify-content: space-evenly;
            align-items: center;
            flex-direction: row;
            margin-left: -40%;
            margin-right: 10%;
            height: 50%;
        }}
        .bottom-right .title {{
            font-size: 64px;
            font-weight: bold;
            color: {bottom_title_color};
            margin: 0;
        }}
        .bottom-right .description {{
            font-size: 18px;
            color: {bottom_description_color};
            line-height: 1.6;
            margin: 0;
            width: 50%;
        }}
        .divider {{
            width: 2px;
            height: 100px;
            background-color: {divider_color};
            margin: 0 30px;
        }}
    </style>
</head>
<body>
    <div class="container-fluid h-100">
        <div class="row h-100">
            <div class="col-md-4 d-flex flex-column align-items-center left-section">
                <div class="number">{left_number}</div>
            </div>
            <div class="col-md-8 d-flex flex-column justify-content-between right-section">
                <div class="row top-right">
                    <div class="col-md-6 image-placeholder">{image_placeholder_text_1}</div>
                    <div class="col-md-6 image-placeholder">{image_placeholder_text_2}</div>
                </div>
                <div class="bottom-right">
                    <div class="col-md-4 title">{bottom_title}</div>
                    <div class="divider"></div>
                    <div class="col-md-4 description">
                        {bottom_description}
                    </div>
                </div>
            </div>
        </div>
    </div>

    <!-- Bootstrap JS and dependencies -->
    <script src="https://cdn.jsdelivr.net/npm/bootstrap@5.3.0/dist/js/bootstrap.bundle.min.js"></script>
</body>
</html>"""

    return html_code

def generate_body_slide1(
    title="Professional HTML Slide Body",
    slide_title="2. Importance of Networking:",
    bg_color="#e9f7fe",
    text_bg_color="#ffffff",
    text_color="#2e4e7e",
    keyword_color="#004080",
    image_bg_color="#b0d4f1",
    image_placeholder_text="Image Placeholder",
    font_family="Arial, sans-serif",
    content_paragraph="Networking is crucial for <span class=\"keyword\">personal development</span>. It fosters:",
    list_items=None
):
    """
    Generate a professional HTML slide body with customizable parameters.

    :param title: The title of the HTML document.
    :param slide_title: The title displayed on the slide.
    :param bg_color: Background color of the page.
    :param text_bg_color: Background color of the text container.
    :param text_color: Text color of the slide content.
    :param keyword_color: Color for keywords.
    :param image_bg_color: Background color of the image placeholder.
    :param image_placeholder_text: Text displayed in the image placeholder.
    :param font_family: Font family for the slide content.
    :param content_paragraph: Main paragraph content.
    :param list_items: A list of bullet points to include.
    :return: A string containing the HTML code.
    """
    if list_items is None:
        list_items = [
            "<span class=\"keyword\">Connections</span>: Builds relationships with professionals in various fields, expanding opportunities and access to knowledge.",
            "<span class=\"keyword\">Mentorship</span>: Enhances professional growth through guidance and support from established individuals.",
            "<span class=\"keyword\">Career Advancement</span>: Connects individuals with potential employers, job leads, and career-enhancing resources.",
            "<span class=\"keyword\">Collaboration</span>: Facilitates sharing of ideas, best practices, and projects, leading to innovation and professional growth."
        ]

    list_html = "\n".join(f"<li>{item}</li>" for item in list_items)

    html_code = f"""<!DOCTYPE html>
<html lang=\"en\">
<head>
    <meta charset=\"UTF-8\">
    <meta name=\"viewport\" content=\"width=device-width, initial-scale=1.0\">
    <title>{title}</title>
    <link href=\"https://cdn.jsdelivr.net/npm/bootstrap@5.3.0/dist/css/bootstrap.min.css\" rel=\"stylesheet\">
    <style>
        body, html {{
            height: 100%;
            margin: 0;
            font-family: {font_family};
            background-color: {bg_color};
        }}
        .slide-container {{
            display: flex;
            flex-direction: column;
            justify-content: center;
            align-items: center;
            height: 100%;
            padding: 20px;
        }}
        .slide-title {{
            color: {keyword_color};
            font-size: 3em;
            margin-bottom: 15px;
            font-weight: bolder;
        }}
        .slide-content {{
            display: flex;
            flex-direction: row;
            align-items: flex-start;
            background-color: {text_bg_color};
            padding: 35px;
            border-radius: 12px;
            box-shadow: 0 4px 12px rgba(0, 0, 0, 0.15);
            color: {text_color};
            max-width: 980px;
            width: 100%;
        }}
        .text-content {{
            flex: 2;
            margin-right: 20px;
        }}
        .text-content p {{
            margin-bottom: 15px;
            font-size: 1.4em;
            line-height: 1.6;
        }}
        .text-content ul {{
            list-style: none;
            padding: 0;
        }}
        .text-content li {{
            margin-top: 12px;
            font-size: 1.2em;
        }}
        .keyword {{
            font-weight: bold;
            color: {keyword_color};
        }}
        .image-placeholder {{
            flex: 1;
            background-color: {image_bg_color};
            width: 250px;
            height: 250px;
            border-radius: 10px;
            display: flex;
            justify-content: center;
            align-items: center;
            color: #333333;
            font-size: 1.1em;
            font-weight: bold;
        }}
        @media (max-width: 768px) {{
            .slide-content {{
                flex-direction: column;
                align-items: center;
            }}
            .text-content, .image-placeholder {{
                margin: 0;
                width: 100%;
                max-width: none;
            }}
            .image-placeholder {{
                margin-top: 20px;
                height: 220px;
            }}
        }}
    </style>
    <script src=\"https://polyfill.io/v3/polyfill.min.js?features=es6\"></script>
    <script type=\"text/javascript\" id=\"MathJax-script\" async
            src=\"https://cdn.jsdelivr.net/npm/mathjax@3/es5/tex-mml-chtml.js\"></script>
</head>
<body>
    <div class=\"slide-container\">
        <div class=\"slide-title\">{slide_title}</div>
        <div class=\"slide-content\">
            <div class=\"text-content\">
                <p>{content_paragraph}</p>
                <ul>
                    {list_html}
                </ul>
            </div>
            <div class=\"image-placeholder\">
                {image_placeholder_text}
            </div>
        </div>
    </div>
</body>
</html>"""

    return html_code

def generate_body_slide2(
    title="Power of Goal Setting",
    header_text="The Power of Goal Setting",
    background_color="#faf0e6",
    text_color="#333",
    content_bg_color="#ffffff",
    content_shadow="0 4px 8px rgba(0, 0, 0, 0.1)",
    header_color="#3B5998",
    text_body_color="#2f4f4f",
    highlight_color="#ff4500",
    image_placeholder_text="[Image Placeholder - Proportioned for future use]",
    image_bg_color="#e1e5ea",
    font_family="Arial, sans-serif",
    paragraph_text=(
        "<strong>Goal setting</strong> is a crucial aspect of personal development that empowers individuals to define clear objectives, map out strategies, and cultivate a path toward growth. "
        "It provides direction, motivation, and accountability, helping people focus their efforts, overcome challenges, and achieve their aspirations."
    )
):
    """
    Generate a professional HTML slide body for presenting content on the power of goal setting.

    :param title: The title of the HTML document.
    :param header_text: The main header of the slide.
    :param background_color: The background color of the entire slide.
    :param text_color: The default text color.
    :param content_bg_color: Background color for the content box.
    :param content_shadow: Box shadow for the content container.
    :param header_color: Color of the header text.
    :param text_body_color: Color of the body text.
    :param highlight_color: Color for highlighted text.
    :param image_placeholder_text: Placeholder text for the main image area.
    :param image_bg_color: Background color of the image placeholder.
    :param font_family: The font family to use for all text.
    :param paragraph_text: The content paragraph.
    :return: A string containing the HTML code.
    """

    html_code = f"""<!DOCTYPE html>
<html lang=\"en\">

<head>
    <meta charset=\"UTF-8\">
    <meta name=\"viewport\" content=\"width=device-width, initial-scale=1.0\">
    <title>{title}</title>
    <script src=\"https://polyfill.io/v3/polyfill.min.js?features=es6\"></script>
    <script id=\"MathJax-script\" async src=\"https://cdn.jsdelivr.net/npm/mathjax@3/es5/tex-mml-chtml.js\"></script>
    <link href=\"https://stackpath.bootstrapcdn.com/bootstrap/4.5.2/css/bootstrap.min.css\" rel=\"stylesheet\">
    <style>
        body,
        html {{
            height: 100%;
            margin: 0;
            font-family: {font_family};
            background-color: {background_color};
        }}

        .slide-container {{
            display: flex;
            justify-content: center;
            align-items: center;
            height: 100vh;
            background-color: {background_color};
        }}

        .content-box {{
            display: flex;
            flex-direction: row;
            background-color: {content_bg_color};
            border-radius: 10px;
            box-shadow: {content_shadow};
            width: 85%;
            padding: 20px;
            color: {text_color};
        }}

        .text-section {{
            flex: 1;
            padding: 20px;
            margin-right: 20px;
            color: {text_body_color};
        }}

        .image-box {{
            flex: 1;
            display: flex;
            justify-content: center;
            align-items: center;
            background-color: {image_bg_color};
            border-radius: 10px;
        }}

        h1 {{
            color: {header_color};
            font-size: 2.5em;
        }}

        p {{
            font-size: 1.2em;
            line-height: 1.6em;
        }}

        strong {{
            color: {highlight_color};
            font-weight: bold;
        }}
    </style>
</head>

<body>
    <div class=\"slide-container\">
        <div class=\"content-box\">
            <div class=\"text-section\">
                <h1>{header_text}</h1>
                <p>{paragraph_text}</p>
            </div>
            <div class=\"image-box\">
                <p style=\"color: #777;\">{image_placeholder_text}</p>
            </div>
        </div>
    </div>
</body>

</html>"""

    return html_code

def generate_body_slide3(
    title="The Future of Creative Industries: Digital Transformation and Immersive Experiences",
    header_text="The Future of Creative Industries: Digital Transformation and Immersive Experiences",
    background_gradient=("#4facfe", "#00c6ff"),
    content_bg_opacity=0.8,
    content_shadow="0 4px 8px rgba(0, 0, 0, 0.1)",
    header_color="#333333",
    text_body_color="#333333",
    highlight_color="#ff4500",
    image_placeholder_text="Image Placeholder",
    image_bg_color="#cccccc",
    font_family="Arial, sans-serif",
    paragraph_text=(
        "The future of the <span class=\"bold\">creative industries</span> lies at the intersection of digital technology and immersive experiences. "
        "<span class=\"bold\">Virtual reality</span>, <span class=\"bold\">augmented reality</span>, and <span class=\"bold\">artificial intelligence</span> are revolutionizing how we create, consume, and interact with art, entertainment, and design. "
        "From immersive exhibitions to <span class=\"bold\">AI-powered storytelling</span>, these technologies are pushing the boundaries of imagination and engagement, opening up new possibilities for creativity and innovation."
    )
):
    """
    Generate a professional HTML slide body for presenting content on digital transformation and immersive experiences in the creative industries.

    :param title: The title of the HTML document.
    :param header_text: The main header of the slide.
    :param background_gradient: Gradient colors for the background.
    :param content_bg_opacity: Opacity for the content box background.
    :param content_shadow: Box shadow for the content container.
    :param header_color: Color of the header text.
    :param text_body_color: Color of the body text.
    :param highlight_color: Color for highlighted text.
    :param image_placeholder_text: Placeholder text for the main image area.
    :param image_bg_color: Background color of the image placeholder.
    :param font_family: The font family to use for all text.
    :param paragraph_text: The content paragraph.
    :return: A string containing the HTML code.
    """

    gradient_css = f"background: linear-gradient(to right, {background_gradient[0]}, {background_gradient[1]});"

    html_code = f"""<!DOCTYPE html>
<html lang=\"en\">

<head>
    <meta charset=\"UTF-8\">
    <meta name=\"viewport\" content=\"width=device-width, initial-scale=1.0\">
    <title>{title}</title>
    <link href=\"https://cdn.jsdelivr.net/npm/bootstrap@5.3.0-alpha1/dist/css/bootstrap.min.css\" rel=\"stylesheet\">
    <style>
        body, html {{
            margin: 0;
            padding: 0;
            height: 100%;
            width: 100%;
            {gradient_css}
            color: #ffffff;
            font-family: {font_family};
        }}

        .custom-background {{
            display: flex;
            justify-content: center;
            align-items: center;
            min-height: 100vh;
            {gradient_css}
        }}

        .content-container {{
            background-color: rgba(255, 255, 255, {content_bg_opacity});
            border-radius: 10px;
            padding: 30px;
            box-shadow: {content_shadow};
            max-width: 800px;
            width: 90%;
            text-align: center;
        }}

        .custom-title {{
            color: {header_color};
            font-weight: bold;
            font-size: 2rem;
            margin-bottom: 15px;
        }}

        .custom-text {{
            font-size: 1.1rem;
            color: {text_body_color};
            margin-bottom: 20px;
        }}

        .bold {{
            font-weight: bold;
        }}

        .image-placeholder {{
            background-color: {image_bg_color};
            width: 100%;
            height: 200px;
            max-width: 300px;
            display: flex;
            align-items: center;
            justify-content: center;
            color: #333;
            border-radius: 5px;
            margin: auto;
        }}
    </style>
</head>
<body>
    <div class=\"custom-background\">
        <div class=\"content-container\">
            <h1 class=\"custom-title\">{header_text}</h1>
            <div class=\"custom-text\">
                {paragraph_text}
            </div>
            <div class=\"d-flex justify-content-center\">
                <div class=\"image-placeholder\">
                    <span>{image_placeholder_text}</span>
                </div>
            </div>
        </div>
    </div>
</body>
</html>"""

    return html_code

In [ ]:
# @title Function Descriptions
def get_function_by_name(name):
    if name == "generate_split_layout_slide1":
        return generate_split_layout_slide1
    elif name == "generate_split_content_slide2":
        return generate_split_layout_slide2
    elif name == "generate_body_slide1":
        return generate_body_slide1
    elif name == "generate_body_slide2":
        return generate_body_slide2
    elif name == "generate_body_slide3":
        return generate_body_slide3
    else:
        raise ValueError(f"Function with name '{name}' not found.")



TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "generate_split_layout_slide1",
            "description": "Generate a split layout HTML slide with customizable parameters for left and right sections, including background colors, text, image placeholders, and a bottom section with a title and description.",
            "parameters": {
                "type": "object",
                "properties": {
                    "left_bg_color": {
                        "type": "string",
                        "description": "Background color of the left section. Defaults to '#2C4B7D'.",
                        "default": "#2C4B7D"
                    },
                    "left_text_color": {
                        "type": "string",
                        "description": "Text color of the left section. Defaults to 'white'.",
                        "default": "white"
                    },
                    "left_number": {
                        "type": "string",
                        "description": "The number displayed in the left section. Defaults to '01'.",
                        "default": "01"
                    },
                    "right_bg_color": {
                        "type": "string",
                        "description": "Background color of the right section. Defaults to '#F0F0F0'.",
                        "default": "#F0F0F0"
                    },
                    "image_placeholder_text_1": {
                        "type": "string",
                        "description": "Text displayed in the first image placeholder. Defaults to 'LLM (Large Language Model) logo Placeholder'.",
                        "default": "LLM (Large Language Model) logo Placeholder"
                    },
                    "image_placeholder_text_2": {
                        "type": "string",
                        "description": "Text displayed in the second image placeholder. Defaults to 'LLM Agent Pipeline Placeholder'.",
                        "default": "LLM Agent Pipeline Placeholder"
                    },
                    "bottom_title": {
                        "type": "string",
                        "description": "Title displayed in the bottom right section. Defaults to 'UltraTool'.",
                        "default": "UltraTool"
                    },
                    "bottom_title_color": {
                        "type": "string",
                        "description": "Color of the bottom title. Defaults to '#2C4B7D'.",
                        "default": "#2C4B7D"
                    },
                    "bottom_description": {
                        "type": "string",
                        "description": "Description text in the bottom right section. Defaults to a description about UltraTool.",
                        "default": "UltraTool cải thiện gọi hàm nhờ lập kế hoạch, xây dựng tools theo yêu cầu phức tạp, và tạo Tool mới khi cần thiết nếu các tool hiện có không đáp ứng được."
                    },
                    "bottom_description_color": {
                        "type": "string",
                        "description": "Color of the bottom description text. Defaults to '#333'.",
                        "default": "#333"
                    },
                    "divider_color": {
                        "type": "string",
                        "description": "Color of the divider in the bottom right section. Defaults to '#000'.",
                        "default": "#000"
                    },
                    "font_family": {
                        "type": "string",
                        "description": "Font family for the slide content. Defaults to 'Arial, sans-serif'.",
                        "default": "Arial, sans-serif"
                    }
                },
                "required": []
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "generate_split_layout_slide2",
            "description": "Generate a split-content HTML slide with customizable parameters for the left section (title, subtitle, dots, colors), the right section (image source and alt text), and the decorative container (background color).",
            "parameters": {
                "type": "object",
                "properties": {
                    "left_bg_color": {
                        "type": "string",
                        "description": "Background color of the left section. Defaults to '#F8F8F8'."
                    },
                    "left_title": {
                        "type": "string",
                        "description": "Title text in the left section. Defaults to 'Báo cáo VNPT-AI về bài toán Function Calling'."
                    },
                    "left_title_color": {
                        "type": "string",
                        "description": "Color of the title in the left section. Defaults to '#2C4B7D'."
                    },
                    "left_subtitle": {
                        "type": "string",
                        "description": "Subtitle text in the left section. Defaults to 'Báo cáo về các bài báo đã tìm hiểu được'."
                    },
                    "left_subtitle_color": {
                        "type": "string",
                        "description": "Color of the subtitle in the left section. Defaults to '#333'."
                    },
                    "left_dots": {
                        "type": "string",
                        "description": "Text or symbols for the dots section in the left section. Defaults to '•••••'."
                    },
                    "left_dots_color": {
                        "type": "string",
                        "description": "Color of the dots in the left section. Defaults to '#2C4B7D'."
                    },
                    "right_bg_color": {
                        "type": "string",
                        "description": "Background color of the right section. Defaults to '#F8F8F8'."
                    },
                    "right_image_src": {
                        "type": "string",
                        "description": "Source path for the image in the right section. Defaults to 'path-to-your-image/LLM-image.png'."
                    },
                    "right_image_alt": {
                        "type": "string",
                        "description": "Alt text for the image in the right section. Defaults to 'LLM Image'."
                    },
                    "decor_bg_color": {
                        "type": "string",
                        "description": "Background color of the decorative container. Defaults to '#2C4B7D'."
                    },
                    "font_family": {
                        "type": "string",
                        "description": "Font family for the slide content. Defaults to 'Arial, sans-serif'."
                    }
                },
                "required": []
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "generate_body_slide1",
            "description": "Generate a professional HTML slide body with customizable content and styling. Use this function when you need to create a slide with a title, a main paragraph, a list of bullet points, and an image placeholder. The slide is designed for professional presentations and supports customization of colors, fonts, and content.",
            "parameters": {
                "type": "object",
                "properties": {
                    "title": {
                        "type": "string",
                        "description": "The title of the HTML document. Defaults to 'Professional HTML Slide Body'.",
                        "default": "Professional HTML Slide Body"
                    },
                    "slide_title": {
                        "type": "string",
                        "description": "The title displayed on the slide. Defaults to '2. Importance of Networking:'.",
                        "default": "2. Importance of Networking:"
                    },
                    "bg_color": {
                        "type": "string",
                        "description": "Background color of the page. Defaults to '#e9f7fe'.",
                        "default": "#e9f7fe"
                    },
                    "text_bg_color": {
                        "type": "string",
                        "description": "Background color of the text container. Defaults to '#ffffff'.",
                        "default": "#ffffff"
                    },
                    "text_color": {
                        "type": "string",
                        "description": "Text color of the slide content. Defaults to '#2e4e7e'.",
                        "default": "#2e4e7e"
                    },
                    "keyword_color": {
                        "type": "string",
                        "description": "Color for keywords in the slide content. Defaults to '#004080'.",
                        "default": "#004080"
                    },
                    "image_bg_color": {
                        "type": "string",
                        "description": "Background color of the image placeholder. Defaults to '#b0d4f1'.",
                        "default": "#b0d4f1"
                    },
                    "image_placeholder_text": {
                        "type": "string",
                        "description": "Text displayed in the image placeholder. Defaults to 'Image Placeholder'.",
                        "default": "Image Placeholder"
                    },
                    "font_family": {
                        "type": "string",
                        "description": "Font family for the slide content. Defaults to 'Arial, sans-serif'.",
                        "default": "Arial, sans-serif"
                    },
                    "content_paragraph": {
                        "type": "string",
                        "description": "Main paragraph content of the slide. Defaults to a paragraph about the importance of networking.",
                        "default": "Networking is crucial for <span class=\"keyword\">personal development</span>. It fosters:"
                    },
                    "list_items": {
                        "type": "array",
                        "items": {
                            "type": "string"
                        },
                        "description": "A list of bullet points to include in the slide. Each item should be a string. If not provided, defaults to a list about networking benefits.",
                        "default": [
                            "<span class=\"keyword\">Connections</span>: Builds relationships with professionals in various fields, expanding opportunities and access to knowledge.",
                            "<span class=\"keyword\">Mentorship</span>: Enhances professional growth through guidance and support from established individuals.",
                            "<span class=\"keyword\">Career Advancement</span>: Connects individuals with potential employers, job leads, and career-enhancing resources.",
                            "<span class=\"keyword\">Collaboration</span>: Facilitates sharing of ideas, best practices, and projects, leading to innovation and professional growth."
                        ]
                    }
                },
                "required": []
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "generate_body_slide2",
            "description": "Generate a professional HTML slide body with customizable layout, colors, and content. Use this function when you need to create a visually appealing slide with a header, a content paragraph, and an image placeholder. The function supports customization of background colors, text colors, fonts, and content, making it ideal for creating presentation slides.",
            "parameters": {
                "type": "object",
                "properties": {
                    "title": {
                        "type": "string",
                        "description": "The title of the HTML document. Defaults to 'Power of Goal Setting'.",
                        "default": "Power of Goal Setting"
                    },
                    "header_text": {
                        "type": "string",
                        "description": "The main header text displayed on the slide. Defaults to 'The Power of Goal Setting'.",
                        "default": "The Power of Goal Setting"
                    },
                    "background_color": {
                        "type": "string",
                        "description": "The background color of the entire slide. Defaults to '#faf0e6'.",
                        "default": "#faf0e6"
                    },
                    "text_color": {
                        "type": "string",
                        "description": "The default text color for the slide. Defaults to '#333'.",
                        "default": "#333"
                    },
                    "content_bg_color": {
                        "type": "string",
                        "description": "The background color of the content box. Defaults to '#ffffff'.",
                        "default": "#ffffff"
                    },
                    "content_shadow": {
                        "type": "string",
                        "description": "The box shadow for the content container. Defaults to '0 4px 8px rgba(0, 0, 0, 0.1)'.",
                        "default": "0 4px 8px rgba(0, 0, 0, 0.1)"
                    },
                    "header_color": {
                        "type": "string",
                        "description": "The color of the header text. Defaults to '#3B5998'.",
                        "default": "#3B5998"
                    },
                    "text_body_color": {
                        "type": "string",
                        "description": "The color of the body text. Defaults to '#2f4f4f'.",
                        "default": "#2f4f4f"
                    },
                    "highlight_color": {
                        "type": "string",
                        "description": "The color for highlighted text (e.g., bold text). Defaults to '#ff4500'.",
                        "default": "#ff4500"
                    },
                    "image_placeholder_text": {
                        "type": "string",
                        "description": "The placeholder text for the image area. Defaults to '[Image Placeholder - Proportioned for future use]'.",
                        "default": "[Image Placeholder - Proportioned for future use]"
                    },
                    "image_bg_color": {
                        "type": "string",
                        "description": "The background color of the image placeholder. Defaults to '#e1e5ea'.",
                        "default": "#e1e5ea"
                    },
                    "font_family": {
                        "type": "string",
                        "description": "The font family for all text in the slide. Defaults to 'Arial, sans-serif'.",
                        "default": "Arial, sans-serif"
                    },
                    "paragraph_text": {
                        "type": "string",
                        "description": "The main content paragraph for the slide. Defaults to a placeholder paragraph about goal setting.",
                        "default": "<strong>Goal setting</strong> is a crucial aspect of personal development that empowers individuals to define clear objectives, map out strategies, and cultivate a path toward growth. It provides direction, motivation, and accountability, helping people focus their efforts, overcome challenges, and achieve their aspirations."
                    }
                },
                "required": []
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "generate_body_slide3",
            "description": "Generate a professional HTML slide body with a gradient background, a centered content box, and a single image placeholder. Use this function when you need to create a visually appealing slide with a header, a content paragraph, and one image placeholder. The function supports customization of background gradients, text colors, fonts, and content, making it ideal for creating modern presentation slides.",
            "parameters": {
                "type": "object",
                "properties": {
                    "title": {
                        "type": "string",
                        "description": "The title of the HTML document. Defaults to 'The Future of Creative Industries: Digital Transformation and Immersive Experiences'.",
                        "default": "The Future of Creative Industries: Digital Transformation and Immersive Experiences"
                    },
                    "header_text": {
                        "type": "string",
                        "description": "The main header text displayed on the slide. Defaults to 'The Future of Creative Industries: Digital Transformation and Immersive Experiences'.",
                        "default": "The Future of Creative Industries: Digital Transformation and Immersive Experiences"
                    },
                    "background_gradient": {
                        "type": "array",
                        "items": {
                            "type": "string"
                        },
                        "description": "Gradient colors for the background, provided as an array of two color codes. Defaults to ['#4facfe', '#00c6ff'].",
                        "default": ["#4facfe", "#00c6ff"]
                    },
                    "content_bg_opacity": {
                        "type": "number",
                        "description": "Opacity for the content box background. Defaults to 0.8.",
                        "default": 0.8
                    },
                    "content_shadow": {
                        "type": "string",
                        "description": "Box shadow for the content container. Defaults to '0 4px 8px rgba(0, 0, 0, 0.1)'.",
                        "default": "0 4px 8px rgba(0, 0, 0, 0.1)"
                    },
                    "header_color": {
                        "type": "string",
                        "description": "Color of the header text. Defaults to '#333333'.",
                        "default": "#333333"
                    },
                    "text_body_color": {
                        "type": "string",
                        "description": "Color of the body text. Defaults to '#333333'.",
                        "default": "#333333"
                    },
                    "highlight_color": {
                        "type": "string",
                        "description": "Color for highlighted text (e.g., bold text). Defaults to '#ff4500'.",
                        "default": "#ff4500"
                    },
                    "image_placeholder_text": {
                        "type": "string",
                        "description": "Placeholder text for the image area. Defaults to 'Image Placeholder'.",
                        "default": "Image Placeholder"
                    },
                    "image_bg_color": {
                        "type": "string",
                        "description": "Background color of the image placeholder. Defaults to '#cccccc'.",
                        "default": "#cccccc"
                    },
                    "font_family": {
                        "type": "string",
                        "description": "The font family for all text in the slide. Defaults to 'Arial, sans-serif'.",
                        "default": "Arial, sans-serif"
                    },
                    "paragraph_text": {
                        "type": "string",
                        "description": "The main content paragraph for the slide. Defaults to a placeholder paragraph about digital transformation and immersive experiences.",
                        "default": "The future of the <span class=\"bold\">creative industries</span> lies at the intersection of digital technology and immersive experiences. <span class=\"bold\">Virtual reality</span>, <span class=\"bold\">augmented reality</span>, and <span class=\"bold\">artificial intelligence</span> are revolutionizing how we create, consume, and interact with art, entertainment, and design. From immersive exhibitions to <span class=\"bold\">AI-powered storytelling</span>, these technologies are pushing the boundaries of imagination and engagement, opening up new possibilities for creativity and innovation."
                    }
                },
                "required": []
            }
        }
    },
]

In [ ]:
from docx import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Step 1: Extract text from the .docx file
def extract_text_from_docx(file_path):
    """
    Extracts text from a .docx file.
    """
    doc = Document(file_path)
    text = []
    for para in doc.paragraphs:
        if para.text.strip():  # Ignore empty paragraphs
            text.append(para.text)
    return "\n".join(text)

# Step 2: Split text into chunks using LangChain
def split_text_into_chunks(text, chunk_size=300, chunk_overlap=50):
    """
    Splits text into chunks using LangChain's RecursiveCharacterTextSplitter.
    """
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
    )
    chunks = text_splitter.split_text(text)
    return chunks

# Input .docx file
docx_file = "test.docx"  # Replace with your .docx file path
# Convert .docx to slides
chunks = split_text_into_chunks(extract_text_from_docx(docx_file), chunk_size=300, chunk_overlap=50)

for chunk in chunks:
    print(chunk)
    print("-" * 100)

Người Việt Nam nên sử dụng AI như thế nào?
Mở đầu
Chào hỏi và giới thiệu bản thân
Nguyễn Minh Quang
Giới thiệu chủ đề thuyết trình
----------------------------------------------------------------------------------------------------
Tầm quan trọng của AI trong thời đại 4.0: AI (Artificial Intelligence) dù mới có quãng thời gian phát triển muộn hơn nhiều so với các ngành trong mảng Công nghệ thông tin khác ngày nay, nhưng tốc độ phát triển của nó lại vô cùng đáng gờm, vượt xa so với gần như tất cả các ngành khác. Trong đó, có
----------------------------------------------------------------------------------------------------
với gần như tất cả các ngành khác. Trong đó, có một mảng con với tốc độ phát triển vượt xa hơn nhiều so với các mảng con khác trong trí tuệ nhân tạo đó là Generative AI (AI tạo sinh) với khả năng thu thập dữ liệu có sẵn để tạo ra dữ liệu mới, nó phá bỏ hoàn toàn quan niệm trước rằng ta chỉ có thể
-----------------------------------------------------------------------

In [ ]:
# @title Content Generator
# Input text
text = """
Người Việt Nam nên sử dụng AI như thế nào?
Mở đầu
Chào hỏi và giới thiệu bản thân
Nguyễn Minh Quang
Giới thiệu chủ đề thuyết trình
----------------------------------------------------------------------------------------------------
Tầm quan trọng của AI trong thời đại 4.0: AI (Artificial Intelligence) dù mới có quãng thời gian phát triển muộn hơn nhiều so với các ngành trong mảng Công nghệ thông tin khác ngày nay, nhưng tốc độ phát triển của nó lại vô cùng đáng gờm, vượt xa so với gần như tất cả các ngành khác. Trong đó, có
----------------------------------------------------------------------------------------------------
với gần như tất cả các ngành khác. Trong đó, có một mảng con với tốc độ phát triển vượt xa hơn nhiều so với các mảng con khác trong trí tuệ nhân tạo đó là Generative AI (AI tạo sinh) với khả năng thu thập dữ liệu có sẵn để tạo ra dữ liệu mới, nó phá bỏ hoàn toàn quan niệm trước rằng ta chỉ có thể
----------------------------------------------------------------------------------------------------
bỏ hoàn toàn quan niệm trước rằng ta chỉ có thể sử dụng AI trong việc phân tích dữ liệu, chuyển văn bản sang giọng nói và ngược lại hay hỗ trợ con người thực hiện các tác vụ lặp lại mà giờ đây nó đã có thể hoàn toàn trò chuyện, suy nghĩ và hoàn thành nhiều tác vụ mà con người đưa ra chỉ bằng 1 câu
----------------------------------------------------------------------------------------------------
nhiều tác vụ mà con người đưa ra chỉ bằng 1 câu nói hay 1 đoạn prompt ngắn.
----------------------------------------------------------------------------------------------------
Mục tiêu của buổi thuyết trình: Khơi dậy sự tò mò, quyết tâm phát triển và làm mới bản thân thông qua việc ứng dụng AI vào đời sống cá nhân.
Phần 1: AI là gì?
Định nghĩa AI (Trí tuệ nhân tạo)
----------------------------------------------------------------------------------------------------
Định nghĩa AI (Trí tuệ nhân tạo)
Hãy tưởng tượng ta có thể dạy một chiếc máy tính làm những việc mà thường cần đến trí thông minh của con người. Đó chính là trí tuệ nhân tạo hay AI. AI giống như việc ta trao cho máy tính một bộ não để nó có thể suy nghĩ và đưa ra quyết định. Ví dụ:
----------------------------------------------------------------------------------------------------
Làm các công việc: Hãy nghĩ đến những việc như nhận diện khuôn mặt của một người bạn trong bức ảnh, nói chuyện với trợ lý ảo như Siri hoặc Alexa, hoặc thậm chí chơi các trò chơi như cờ vua hay game điện tử. AI giúp máy tính thực hiện các công việc này mà không cần phải chỉ dẫn chi tiết từng bước.
----------------------------------------------------------------------------------------------------
Học hỏi từ kinh nghiệm: Giống như cách chúng ta học từ kinh nghiệm, AI cũng có thể học. Ví dụ, nếu ta cho máy tính xem hàng ngàn bức ảnh của mèo, nó có thể học cách nhận diện một con mèo trong bức ảnh mới. Đây gọi là "học máy".
----------------------------------------------------------------------------------------------------
Ra quyết định: AI có thể đưa ra quyết định dựa trên dữ liệu. Chẳng hạn, nó có thể đề xuất bộ phim mà ta có thể thích xem tiếp theo trên Netflix bằng cách xem xét những gì ta đã xem trước đó và những gì người khác có sở thích tương tự đã thích.
Các loại AI
----------------------------------------------------------------------------------------------------
Các loại AI
AI hẹp: Loại AI này giỏi làm một việc cụ thể. Ví dụ như một chương trình chỉ chơi cờ vua hoặc chỉ đề xuất các bài hát mà ta có thể thích. Nó không thể làm gì ngoài nhiệm vụ cụ thể của mình.
----------------------------------------------------------------------------------------------------
AI toàn diện: Loại AI này sẽ thông minh như con người ở nhiều khía cạnh khác nhau. Nó có thể làm nhiều việc khác nhau, giống như con người, có thể suy nghĩ và đưa ra phương hướng giải quyết cho bất kỳ vấn đề nào.
Phần 2: AI có tiềm năng gì?
Lợi ích của AI
Tổng quan:
----------------------------------------------------------------------------------------------------
Lợi ích của AI
Tổng quan:
AI giúp ta thực hiện các công việc lặp đi lặp lại với độ chính xác gần tương đương, đôi khi là hơn so với việc con người xử lý.
----------------------------------------------------------------------------------------------------
AI hỗ trợ trong việc chuyển hóa dữ liệu, và phân tích dữ liệu, như dịch văn bản đa ngôn ngữ, chuyển hóa từ giọng nói sang chữ viết, nhận diện giọng nói, chuyển từ hình ảnh sang văn bản,... Chúng giúp cải thiện năng suất làm việc của ta bằng cách tự động hóa 1 số các tác vụ cơ bản lặp đi lặp lại mà
----------------------------------------------------------------------------------------------------
động hóa 1 số các tác vụ cơ bản lặp đi lặp lại mà trước đây cần con người xử lý, giúp tăng năng suất và cải thiện số lượng công việc mà 1 con người có thể làm trong 1 ngày.
----------------------------------------------------------------------------------------------------
Tuy nhiên, cho tới thời điểm hiện tại AI lại tiếp tục nổi lên như một cơn bão khi có sự xuất hiện của các chatbot tân tiến với khả năng trò chuyện và truyền đạt kiến thức, hỗ trợ các tác vụ cơ bản. AI hiện tại không chỉ giúp thực hiện các tác vụ lặp lại mà bây giờ, dựa vào khả năng tích lũy kiến
----------------------------------------------------------------------------------------------------
lại mà bây giờ, dựa vào khả năng tích lũy kiến thức, kinh nghiệm từ những quá trình huấn luyện, nó có thể đóng vai trò như một cầu nối truyền đạt kiến thức đến cho ta từ 1 nguồn dữ liệu khổng lồ đến từ rất nhiều mảng và nhiều ngành khác nhau. Nó hỗ trợ cho ta cái nhìn tổng quan hơn, tìm được những
----------------------------------------------------------------------------------------------------
trợ cho ta cái nhìn tổng quan hơn, tìm được những ngóc ngách quan trọng mà chỉ với vốn kiến thức của bản thân là không đủ để nhìn thấy.
----------------------------------------------------------------------------------------------------
Cụ thể:
Trong mảng Lập trình, một lập trình viên thông thường có thể hoàn thành được trung bình một dự án nhỏ trong 1 tháng, nhưng với sự trợ giúp của AI, hỗ trợ cho việc cải tiến, gợi ý code có thể giúp lập trình viên hoàn thành dự án nhanh gấp 2 có khi là gấp 3 lần thông thường.
----------------------------------------------------------------------------------------------------
Trong mảng Tiếp thị, AI có thể hỗ trợ, phân tích và đưa ra cái nhìn tổng quan và dự đoán hướng đi hay tự động hóa việc viết bài quảng cáo cho doanh nghiệp,...
----------------------------------------------------------------------------------------------------
Ngay cả những tiệm tạp hóa hay các tiệm buôn bán nhỏ lẻ cũng có thể sử dụng AI để phân tích chiến lược bán sản phẩm như tại sao sản phẩm này hiện tại được nhiều người mua như vậy, sản phẩm này có thể sử dụng cho những mục đích gì và sản phẩm này thường được sử dụng cùng với những sản phẩm nào khác?
----------------------------------------------------------------------------------------------------
Các ví dụ về AI trong cuộc sống hàng ngày
Trợ lý ảo: Siri, Google Assistant, Alexa.
Dịch vụ giải trí: Gợi ý phim trên Netflix, gợi ý nhạc trên Spotify.
Sức khỏe: Ứng dụng theo dõi sức khỏe như Fitbit, Apple Health.
----------------------------------------------------------------------------------------------------
Giáo dục: Các nền tảng học tập trực tuyến sử dụng AI để cá nhân hóa trải nghiệm học tập như Khan Academy, Duolingo.
Tài chính: Các ứng dụng quản lý tài chính cá nhân như Mint sử dụng AI để phân tích chi tiêu và gợi ý tiết kiệm.
----------------------------------------------------------------------------------------------------
Giao thông: Hệ thống dẫn đường như Google Maps, Waze sử dụng AI để dự báo tình trạng giao thông và gợi ý lộ trình tối ưu.
Phần 3: Tại sao người Việt Nam nên ứng dụng AI vào cuộc sống?
Cơ hội và thách thức đối với Việt Nam
Cơ hội:
----------------------------------------------------------------------------------------------------
Tiềm năng kinh tế: AI có thể tăng năng suất lao động và thúc đẩy sự phát triển kinh tế, giảm tải lượng công việc lặp đi lặp lại cho người lao động. Với sự hỗ trợ của AI, các doanh nghiệp có thể tối ưu hóa quy trình sản xuất, nâng cao hiệu quả và giảm chi phí, từ đó tạo ra lợi thế cạnh tranh trên
----------------------------------------------------------------------------------------------------
chi phí, từ đó tạo ra lợi thế cạnh tranh trên thị trường.
----------------------------------------------------------------------------------------------------
Tri thức: AI mở ra kho tàng tri thức khổng lồ mà ai cũng có thể tiếp cận, giúp nâng cao kỹ năng và kiến thức cá nhân. Việc áp dụng AI trong giáo dục và đào tạo không chỉ nâng cao chất lượng học tập mà còn tạo cơ hội cho mọi người phát triển bản thân, góp phần xây dựng một xã hội tri thức.
----------------------------------------------------------------------------------------------------
Thách thức:
----------------------------------------------------------------------------------------------------
Thiếu nhân lực chuyên môn và cơ sở hạ tầng: Việt Nam hiện đang thiếu nhân lực có chuyên môn cao trong lĩnh vực AI, điều này gây khó khăn cho việc triển khai và phát triển các dự án AI. Cần đầu tư mạnh mẽ vào giáo dục và đào tạo để khắc phục vấn đề này. Đồng thời, cần có sự đầu tư và cải thiện cơ sở
----------------------------------------------------------------------------------------------------
Đồng thời, cần có sự đầu tư và cải thiện cơ sở hạ tầng để hỗ trợ cho sự phát triển của AI.
----------------------------------------------------------------------------------------------------
Nhận thức của người dùng: Đa số người dùng Việt Nam chưa thật sự được tiếp cận, tìm hiểu hay nhận ra được sự hữu ích mà AI có thể đem lại. Việc nâng cao nhận thức và kiến thức về AI cho cộng đồng là rất quan trọng để thúc đẩy việc ứng dụng AI rộng rãi.
----------------------------------------------------------------------------------------------------
Phần 4: Hướng dẫn người Việt Nam tiếp cận và học hỏi về AI
Các khóa học và tài liệu học tập về AI
Về hướng học thuật:
Các khóa học trực tuyến (Coursera, edX, Udacity).
Các tài liệu, sách vở về AI (tài liệu tiếng Việt và tiếng Anh).
Về hướng chỉ ứng dụng:
----------------------------------------------------------------------------------------------------
Về hướng chỉ ứng dụng:
Có rất nhiều kênh youtube hay trang facebook hướng dẫn việc sử dụng AI mà không đến mức học thuật, nên bất kể là ai, dù có kiến thức nền về công nghệ thông tin hay là không vẫn có thể thực hiện được. Như kênh Mì AI, Phan Đông Giang, Thái Vân Linh,...
----------------------------------------------------------------------------------------------------
Không cần đi đâu xa ta có thể hỏi chính những mô hình chatbot nổi tiếng hiện nay như chat gpt, claude hay gemini để tìm định hướng cho bản thân.
Các sự kiện và cộng đồng AI tại Việt Nam
Tham gia các hội thảo, sự kiện công nghệ.
Tham gia các cộng đồng AI như Bình Dân Học AI, Mì AI,...
Kết luận
----------------------------------------------------------------------------------------------------
Kết luận
Tóm tắt lại các ý chính
Định nghĩa và các loại hình của AI:
AI là khả năng của máy tính để thực hiện các tác vụ thường cần đến trí tuệ của con người.
AI hẹp (Narrow AI) và AI toàn diện (General AI).
Tiềm năng và lợi ích của AI trong cuộc sống hàng ngày:
----------------------------------------------------------------------------------------------------
AI giúp tự động hóa các công việc lặp đi lặp lại, tăng năng suất và giảm chi phí.
AI trong các lĩnh vực: lập trình, tiếp thị, y tế, giáo dục, tài chính, và giao thông.
Tại sao người Việt Nam nên ứng dụng AI:
Cơ hội tăng trưởng kinh tế và nâng cao chất lượng cuộc sống.
----------------------------------------------------------------------------------------------------
Thách thức về nhân lực và cơ sở hạ tầng cần được khắc phục.
Các phương pháp và tài liệu học tập về AI:
Khóa học trực tuyến (Coursera, edX, Udacity), tài liệu, các chatbot chat gpt, claude, gemini,...
----------------------------------------------------------------------------------------------------
Các kênh trao đổi kiến thức về ứng dụng AI trên nhiều nền tảng như YouTube, trang Facebook (Mì AI, Phan Đông Giang, Thái Vân Linh).
Tầm quan trọng của việc ứng dụng AI
AI không chỉ là công cụ hỗ trợ mà còn là yếu tố chiến lược giúp doanh nghiệp và cá nhân tăng hiệu quả công việc.
----------------------------------------------------------------------------------------------------
Sự phát triển nhanh chóng của AI sẽ mở ra nhiều cơ hội mới trong mọi lĩnh vực.
"""

# Step 1: Split the text by the delimiter "----------------------------------------------------------------------------------------------------"
sections = text.strip().split("----------------------------------------------------------------------------------------------------")

# Step 2: Clean and combine sections to ensure coherence
slide_contents = []
current_slide = ""
for section in sections:
    section = section.strip()
    if not section:
        continue
    # If the current slide is not too long, add the section to it
    if len(current_slide) + len(section) <= 1000:  # Adjust the limit as needed
        current_slide += "\n" + section if current_slide else section
    else:
        # If the current slide is too long, save it and start a new one
        slide_contents.append(current_slide)
        current_slide = section
# Add the last slide
if current_slide:
    slide_contents.append(current_slide)

# Step 3: Print the slide contents
for i, slide in enumerate(slide_contents):
    print(f"Slide {i + 1}:\n{slide}\n")

# Step 4: Save the slide contents to a list
slide_list = slide_contents

Slide 1:
Người Việt Nam nên sử dụng AI như thế nào?
Mở đầu
Chào hỏi và giới thiệu bản thân
Nguyễn Minh Quang
Giới thiệu chủ đề thuyết trình
Tầm quan trọng của AI trong thời đại 4.0: AI (Artificial Intelligence) dù mới có quãng thời gian phát triển muộn hơn nhiều so với các ngành trong mảng Công nghệ thông tin khác ngày nay, nhưng tốc độ phát triển của nó lại vô cùng đáng gờm, vượt xa so với gần như tất cả các ngành khác. Trong đó, có
với gần như tất cả các ngành khác. Trong đó, có một mảng con với tốc độ phát triển vượt xa hơn nhiều so với các mảng con khác trong trí tuệ nhân tạo đó là Generative AI (AI tạo sinh) với khả năng thu thập dữ liệu có sẵn để tạo ra dữ liệu mới, nó phá bỏ hoàn toàn quan niệm trước rằng ta chỉ có thể

Slide 2:
bỏ hoàn toàn quan niệm trước rằng ta chỉ có thể sử dụng AI trong việc phân tích dữ liệu, chuyển văn bản sang giọng nói và ngược lại hay hỗ trợ con người thực hiện các tác vụ lặp lại mà giờ đây nó đã có thể hoàn toàn trò chuyện, suy nghĩ và hoàn thành nhi

In [ ]:
demand_prompt = """
Create a slide that match the following content, choose a function when you think it is best suit with the content:
You can also choose the function based on the previous slide content and function call, if there is no previous content or function call, which means this is the first slide.

Previous slide content:
{}

Previous function call:
{}

Current slide content:
{}

Current function call:
"""


def get_html_slide(pre_slide_content, pre_function_call, slide_content):
    MESSAGES = [
        {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."},
        {"role": "user",  "content": demand_prompt.format(pre_slide_content, pre_function_call, slide_content)},
    ]


    # print(demand_prompt.format(pre_slide_content, pre_function_call, slide_content))

    tools = TOOLS
    messages = MESSAGES[:]

    text = tokenizer.apply_chat_template(messages, tools=tools, add_generation_prompt=True, tokenize=False)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=512)
    output_text = tokenizer.batch_decode(outputs)[0][len(text):]
    return output_text

slide_function_calling_list = []
pre_slide_content = ""
pre_function_call = ""

for slide_content in slide_list:
    html_slide_call = get_html_slide(pre_slide_content, pre_function_call, slide_content)
    pre_slide_content = slide_content
    pre_function_call = html_slide_call
    print(html_slide_call)
    print("-" * 100)
    slide_function_calling_list.append(html_slide_call)

<tool_call>
{"name": "generate_body_slide1", "arguments": {"title": "Người Việt Nam nên sử dụng AI như thế nào?", "slide_title": "Mở đầu", "bg_color": "#e9f7fe", "text_bg_color": "#ffffff", "text_color": "#2e4e7e", "keyword_color": "#004080", "image_bg_color": "#b0d4f1", "image_placeholder_text": "Image Placeholder", "font_family": "Arial, sans-serif", "content_paragraph": "Nguyễn Minh Quang giới thiệu về tầm quan trọng của AI trong thời đại 4.0: AI dù mới có quãng thời gian phát triển muộn hơn nhiều so với các ngành trong mảng Công nghệ thông tin khác ngày nay, nhưng tốc độ phát triển của nó lại vô cùng đáng gờm, vượt xa so với gần như tất cả các ngành khác.", "list_items": ["Generative AI (AI tạo sinh) với khả năng thu thập dữ liệu có sẵn để tạo ra dữ liệu mới, nó phá bỏ hoàn toàn quan niệm trước rằng ta chỉ có thể"]}}
</tool_call><|im_end|>
----------------------------------------------------------------------------------------------------


OutOfMemoryError: CUDA out of memory. Tried to allocate 1.86 GiB. GPU 0 has a total capacity of 14.75 GiB of which 969.06 MiB is free. Process 2362 has 13.80 GiB memory in use. Of the allocated memory 11.01 GiB is allocated by PyTorch, and 2.66 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
import re
import json

def try_parse_tool_calls(content: str):
    """Try parse the tool calls."""
    tool_calls = []
    offset = 0
    for i, m in enumerate(re.finditer(r"<tool_call>\n(.+)?\n</tool_call>", content)):
        if i == 0:
            offset = m.start()
        try:
            func = json.loads(m.group(1))
            tool_calls.append({"type": "function", "function": func})
            if isinstance(func["arguments"], str):
                func["arguments"] = json.loads(func["arguments"])
        except json.JSONDecodeError as e:
            print(f"Failed to parse tool calls: the content is {m.group(1)} and {e}")
            pass
    if tool_calls:
        if offset > 0 and content[:offset].strip():
            c = content[:offset]
        else:
            c = ""
        return {"role": "assistant", "content": c, "tool_calls": tool_calls}
    return {"role": "assistant", "content": re.sub(r"<\|im_end\|>$", "", content)}

messages.append(try_parse_tool_calls(output_text))

if tool_calls := messages[-1].get("tool_calls", None):
    for tool_call in tool_calls:
        if fn_call := tool_call.get("function"):
            fn_name: str = fn_call["name"]
            fn_args: dict = fn_call["arguments"]

            fn_res: str = json.dumps(get_function_by_name(fn_name)(**fn_args))

            messages.append({
                "role": "tool",
                "name": fn_name,
                "content": fn_res,
            })

# Function to capture the HTML slide as an image
def capture_slide_image(html_content):
    options = Options()
    options.headless = True
    driver = gs.UndetectedChrome(options=options)

    # Save HTML to a temporary file
    temp_html_path = 'temporal_slide.html'
    with open(temp_html_path, 'w') as file:
        file.write(html_content)

    # Load the HTML file in Selenium
    driver.get(f"file://{os.path.abspath(temp_html_path)}")
    driver.set_window_size(1920, 1080)

    # Capture the screenshot
    screenshot = driver.get_screenshot_as_png()
    driver.quit()

    # Open image and resize to the desired dimensions (900x500)
    image = Image.open(io.BytesIO(screenshot))
    resized_image = image.resize((900, 500), Image.Resampling.LANCZOS)

    return resized_image

In [ ]:
for i, chunk in enumerate(chunks[1:]):
    attempts = 0
    df_dict["Raw Content"].append(chunk)
    html_content = ""
    evaluation_response = ""


    while attempts < threholds:
        html_content = generate_html_slide(html_content, chunk, evaluation_response)
        pattern = r'```html\n([\s\S]*?)\n```'
        match = re.search(pattern, html_content)
        if match:
            html_content = match.group(1)
        slide_image = capture_slide_image(html_content)
        evaluation_response = evaluate_slide_image(html_content, slide_image)
        # Extract the evaluation content
        evaluation_content = evaluation_response['choices'][0]['message']['content'].strip()

        if not os.path.exists(f"slide/slide_{i+1}"): os.makedirs(f"slide/slide_{i+1}")
        with open(f"slide/slide_{i+1}/attempt_{attempts}.html", 'w') as file:
            file.write(html_content)
        slide_image.save(f"slide/slide_{i+1}/attempt_{attempts}.png")
        if "The slide is acceptable" in evaluation_content:
            # Save the HTML content and the image
            print(f"Slide {i+1} accepted.")
            break
        else:
            print(f"Slide {i+1} not accepted.")
            attempts += 1

        if attempts == threholds:
            html_content = generate_html_slide(html_content, chunk, evaluation_response)
            pattern = r'```html\n([\s\S]*?)\n```'
            match = re.search(pattern, html_content)
            if match:
                html_content = match.group(1)
            slide_image = capture_slide_image(html_content)
            if not os.path.exists(f"slide/slide_{i+1}"): os.makedirs(f"slide/slide_{i+1}")
            with open(f"slide/slide_{i+1}/attempt_{attempts}.html", 'w') as file:
                file.write(html_content)
            slide_image.save(f"slide/slide_{i+1}/attempt_{attempts}.png")
            print(f"Slide {i+1} accepted after {threholds} reviews.")

    df_dict["Html Slide"].append(html_content)
    plan_content = generate_plan_from_html(html_content)
    df_dict["Plan"].append(plan_content)